# Shrimp Disease Paper Runner

Run these cells on Kaggle after uploading or cloning the clean paper codebase. The defaults attempt the real stages and rely on script-level resume validation to skip runs with valid final metrics.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = Path('/kaggle/working/shrimp_outputs')
SCREENING_OUTPUT_DIR = Path('/kaggle/working/shrimp_outputs_asl_custom_screening')
LOG_DIR = Path('/kaggle/working/notebook_command_logs')

RUN_PREPARE_REAL_DATASET = True
RUN_STAGE_02_CORE_ABLATION = True
RUN_STAGE_03_YOLO_FAMILY = True
RUN_STAGE_04_LIGHTWEIGHT = True
RUN_STAGE_05_REPORTS = True
RUN_ASL_CUSTOM_SCREENING = True
RUN_VERIFY = True
RUN_COLLECT = True

ALLOW_DELETE_OUTPUTS = False
OVERWRITE_OUTPUTS = False

LOG_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    'project_dir': str(PROJECT_DIR),
    'output_dir': str(OUTPUT_DIR),
    'screening_output_dir': str(SCREENING_OUTPUT_DIR),
    'log_dir': str(LOG_DIR),
    'run_prepare_real_dataset': RUN_PREPARE_REAL_DATASET,
    'run_stage_02_core_ablation': RUN_STAGE_02_CORE_ABLATION,
    'run_stage_03_yolo_family': RUN_STAGE_03_YOLO_FAMILY,
    'run_stage_04_lightweight': RUN_STAGE_04_LIGHTWEIGHT,
    'run_stage_05_reports': RUN_STAGE_05_REPORTS,
    'run_asl_custom_screening': RUN_ASL_CUSTOM_SCREENING,
    'run_verify': RUN_VERIFY,
    'run_collect': RUN_COLLECT,
    'allow_delete_outputs': ALLOW_DELETE_OUTPUTS,
    'overwrite_outputs': OVERWRITE_OUTPUTS,
}, indent=2))


In [ ]:
def run_command(label, args):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log_path = LOG_DIR / f'{timestamp}_{label}.log'
    command = [sys.executable, *args]
    print('\n$ ' + ' '.join(str(part) for part in command))
    with log_path.open('w', encoding='utf-8') as log_file:
        log_file.write('$ ' + ' '.join(str(part) for part in command) + '\n')
        process = subprocess.Popen(
            command,
            cwd=str(PROJECT_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'{label} failed with exit code {code}. See {log_path}')
    return log_path


In [ ]:
if RUN_PREPARE_REAL_DATASET:
    run_command('01_prepare_dataset', ['shrimp_scripts/run_01_prepare_dataset.py', '--output_dir', str(OUTPUT_DIR)])


In [ ]:
if RUN_STAGE_02_CORE_ABLATION:
    run_command('02_core_ablation', ['shrimp_scripts/run_02_train_core_ablation.py', '--output_dir', str(OUTPUT_DIR), '--resume', '--progress'])


In [ ]:
if RUN_STAGE_03_YOLO_FAMILY:
    run_command('03_yolo_family', ['shrimp_scripts/run_03_train_yolo_family.py', '--output_dir', str(OUTPUT_DIR), '--resume', '--progress'])


In [ ]:
if RUN_STAGE_04_LIGHTWEIGHT:
    run_command('04_lightweight_models', ['shrimp_scripts/run_04_train_lightweight_models.py', '--output_dir', str(OUTPUT_DIR), '--resume', '--progress'])


In [ ]:
if RUN_STAGE_05_REPORTS:
    run_command('05_reports_and_xai', ['shrimp_scripts/run_05_generate_reports_and_xai.py', '--output_dir', str(OUTPUT_DIR), '--progress'])


In [ ]:
if RUN_ASL_CUSTOM_SCREENING:
    run_command('screening_prepare_dataset', ['shrimp_scripts/run_01_prepare_dataset.py', '--output_dir', str(SCREENING_OUTPUT_DIR)])
    run_command('screening_train', ['experiments/asl_custom_loss_screening/run_asl_custom_screen.py', '--output_dir', str(SCREENING_OUTPUT_DIR), '--resume', '--progress'])


In [ ]:
if RUN_VERIFY:
    run_command('verify_stage02_resume', ['shrimp_scripts/run_02_train_core_ablation.py', '--output_dir', str(OUTPUT_DIR), '--validate_resume', '--no_progress'])
    run_command('verify_stage03_resume', ['shrimp_scripts/run_03_train_yolo_family.py', '--output_dir', str(OUTPUT_DIR), '--validate_resume', '--no_progress'])
    run_command('verify_stage04_resume', ['shrimp_scripts/run_04_train_lightweight_models.py', '--output_dir', str(OUTPUT_DIR), '--validate_resume', '--no_progress'])
    run_command('verify_screening_resume', ['experiments/asl_custom_loss_screening/run_asl_custom_screen.py', '--output_dir', str(SCREENING_OUTPUT_DIR), '--validate_resume', '--no_progress'])


In [ ]:
if RUN_COLLECT:
    run_command('collect_main_reports', ['shrimp_scripts/run_05_generate_reports_and_xai.py', '--output_dir', str(OUTPUT_DIR), '--progress'])
    run_command('collect_screening_results', ['experiments/asl_custom_loss_screening/collect_asl_custom_results.py', '--output_dir', str(SCREENING_OUTPUT_DIR)])
